In [41]:
import os, shutil

REPO = "/content/AAI-590-Capstone-group2"
if not os.path.exists(REPO):
    os.system(f"git clone https://github.com/TEJSINGH17/AAI-590-Capstone-group2.git {REPO}")
    print("Repo cloned")
else:
    print("Repo exists")

os.system("pip install ultralytics opencv-python-headless imageio imageio-ffmpeg -q")
print("Dependencies ready")

video = f"{REPO}/output_ds_1.mp4"
size  = os.path.getsize(video)/1e6 if os.path.exists(video) else 0
if size > 1:
    print(f"Video ready: {size:.1f} MB")
else:
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded.keys():
        shutil.move(fname, f"{REPO}/{fname}")
print("Setup done!")

Repo exists
Dependencies ready
Video ready: 66.9 MB
Setup done!


In [54]:

import os
REPO = "/content/AAI-590-Capstone-group2"

# Step 1: restore original from git first (removes any previous patches)
os.system(f"cd {REPO} && git checkout omniview_e2e_live.py")
print("Original file restored")

code = open(f"{REPO}/omniview_e2e_live.py").read()

# ── Patch 1: add performance counter variables ──────────────
old1 = "    vel_tracker = VelocityTracker()\n\n    total = 0\n    t0    = time.time()\n    i     = 0"
new1 = """    vel_tracker = VelocityTracker()

    total            = 0
    lisa_total       = 0
    coco_total       = 0
    left_alert_total = 0
    right_alert_total= 0
    fast_app_total   = 0
    frames_with_dets = 0
    frame_times      = []
    t0               = time.time()
    i                = 0"""

# ── Patch 2: start frame timer at top of loop ───────────────
old2 = "        for i in range(max_frames):\n            if _canvas_only:"
new2 = "        for i in range(max_frames):\n            _ft = time.time()\n            if _canvas_only:"

# ── Patch 3: count detections by model/zone ─────────────────
old3 = "                total += 1\n\n            if instruction:"
new3 = """                total += 1
                if model_src == "lisa": lisa_total += 1
                else:                   coco_total += 1
                if bs == "left":        left_alert_total  += 1
                if bs == "right":       right_alert_total += 1
                if fast_approach:       fast_app_total    += 1

            if instruction:"""

# ── Patch 4: record frame render time after writer.write ────
old4 = "            writer.write(frame)\n\n            if i % 100 == 0:"
new4 = """            writer.write(frame)

            frame_times.append((time.time() - _ft) * 1000)
            if len(dets) > 0:
                frames_with_dets += 1

            if i % 100 == 0:"""

# ── Patch 5: save perf data to JSON file ────────────────────
old5 = '        print(f"[Pipeline] MP4 saved: {out_path}"\n              f" ({os.path.getsize(str(out_path))//1024} KB)")'
new5 = '''        try:
            mp4_kb = os.path.getsize(str(out_path)) // 1024
            print(f"[Pipeline] MP4 saved: {out_path} ({mp4_kb:,} KB)")
        except Exception:
            mp4_kb = 0
        try:
            _ft_arr = np.array(frame_times) if frame_times else np.array([0.0])
            _perf = {
                "mode":             args.mode.upper(),
                "resolution":       f"{w} x {h} @ {fps:.0f} fps",
                "total_frames":     i + 1,
                "elapsed_s":        round(elapsed, 2),
                "avg_fps":          round((i+1)/elapsed, 2),
                "avg_render_ms":    round(float(np.mean(_ft_arr)), 2),
                "min_render_ms":    round(float(np.min(_ft_arr)), 2),
                "max_render_ms":    round(float(np.max(_ft_arr)), 2),
                "total_dets":       total,
                "avg_dets_frame":   round(total / max(i+1,1), 2),
                "frames_with_dets": frames_with_dets,
                "det_rate_pct":     round(frames_with_dets / max(i+1,1) * 100, 1),
                "lisa_total":       lisa_total,
                "coco_total":       coco_total,
                "left_alerts":      left_alert_total,
                "right_alerts":     right_alert_total,
                "fast_approach":    fast_app_total,
                "tracked_objects":  len(vel_tracker.history),
                "mp4_kb":           mp4_kb,
            }
            import json as _json
            _perf_path = f"/content/perf_{args.mode}.json"
            with open(_perf_path, "w") as _pf:
                _json.dump(_perf, _pf)
            print(f"[Pipeline] Perf saved -> {_perf_path}")
        except Exception as _e:
            print(f"[Pipeline] Perf save failed: {_e}")'''

# ── Apply all patches ────────────────────────────────────────
patches = [(old1,new1),(old2,new2),(old3,new3),(old4,new4),(old5,new5)]
for idx, (old, new) in enumerate(patches, 1):
    if old in code:
        code = code.replace(old, new)
        print(f" Patch {idx} applied")
    else:
        print(f" Patch {idx} NOT FOUND — re-clone repo and retry")

with open(f"{REPO}/omniview_e2e_live.py", "w") as f:
    f.write(code)

# ── Verify ───────────────────────────────────────────────────
print()
checks = [
    ("Counter vars",       "frame_times      = []"),
    ("Frame timer",        "_ft = time.time()"),
    ("Detection counters", "lisa_total += 1"),
    ("Frame timing",       "frame_times.append"),
    ("Perf save",          "perf_{args.mode}"),
]
all_ok = True
for name, check in checks:
    ok = check in code
    print(f"{'yes' if ok else 'X'}  {name}")
    if not ok:
        all_ok = False

print()
print(" All patches applied — ready to run!" if all_ok else "X Some patches failed — do NOT run pipeline yet")

Original file restored
 Patch 1 applied
 Patch 2 applied
 Patch 3 applied
 Patch 4 applied
 Patch 5 applied

yes  Counter vars
yes  Frame timer
yes  Detection counters
yes  Frame timing
yes  Perf save

 All patches applied — ready to run!


In [48]:
import os
os.chdir("/content")
REPO = "/content/AAI-590-Capstone-group2"

result = __import__('subprocess').run([
    "python3", f"{REPO}/omniview_e2e_live.py",
    "--no-bootstrap", "--mode", "file",
    "--json",   f"{REPO}/runs/hud/ds_1",
    "--video",  f"{REPO}/output_ds_1.mp4",
    "--output", "/content/omniview_live_hud.mp4",
    "--gif-frames", "40", "--gif-fps", "10",
], capture_output=True, text=True)
print(result.stdout[-2000:])

  OmniView AI -- End-to-End Live HUD  v4.0
  AAI-590 Capstone Group 2 | USD
  Victor's omniview_pipeline.py loaded
     get_blind_spot_zones, draw_blind_spot_zones
     get_box_color, draw_alerts
     DetectionPayload, build_message
  Sunitha's VelocityTracker + INSTRUCTION_MAP loaded
  Tej's ds_pipeline.py referenced for Mode 3 (Jetson)
     Cannot import on Colab -- Jetson hardware required

  Mode   : FILE
  Video  : /content/AAI-590-Capstone-group2/output_ds_1.mp4
  JSON   : /content/AAI-590-Capstone-group2/runs/hud/ds_1
  Output : /content/omniview_live_hud.mp4
[Pipeline] Video  : /content/AAI-590-Capstone-group2/output_ds_1.mp4  1920x1080 @ 30.0fps
[Pipeline] JSON   : 1107 frames from /content/AAI-590-Capstone-group2/runs/hud/ds_1
[Pipeline] Total  : 1107 frames
frame=0/1107  dets=6  total=6  0.0fps
frame=100/1107  dets=3  total=705  5.8fps
frame=200/1107  dets=5  total=1240  6.5fps
frame=300/1107  dets=6  total=1803  6.8fps
frame=400/1107  dets=6  total=2369  6.9fps
frame=500/11

In [49]:
import os, subprocess
os.chdir("/content")
REPO = "/content/AAI-590-Capstone-group2"

result = subprocess.run([
    "python3", f"{REPO}/omniview_e2e_live.py",
    "--no-bootstrap", "--mode", "file",
    "--video",  f"{REPO}/output_ds_1.mp4",
    "--output", "/content/omniview_live_hud.mp4",
    "--gif-frames", "40", "--gif-fps", "10",
], capture_output=True, text=True)

print(result.stdout[-2000:])
if result.stderr:
    print("STDERR:", result.stderr[-500:])


  OmniView AI -- End-to-End Live HUD  v4.0
  AAI-590 Capstone Group 2 | USD
  Victor's omniview_pipeline.py loaded
     get_blind_spot_zones, draw_blind_spot_zones
     get_box_color, draw_alerts
     DetectionPayload, build_message
  Sunitha's VelocityTracker + INSTRUCTION_MAP loaded
  Tej's ds_pipeline.py referenced for Mode 3 (Jetson)
     Cannot import on Colab -- Jetson hardware required

  Mode   : FILE
  Video  : /content/AAI-590-Capstone-group2/output_ds_1.mp4
  JSON   : /content/AAI-590-Capstone-group2/runs/hud/ds_1
  Output : /content/omniview_live_hud.mp4
[Pipeline] Video  : /content/AAI-590-Capstone-group2/output_ds_1.mp4  1920x1080 @ 30.0fps
[Pipeline] JSON   : 1107 frames from /content/AAI-590-Capstone-group2/runs/hud/ds_1
[Pipeline] JSON   : 1096 frames from /content/AAI-590-Capstone-group2/runs/hud/ds_2
[Pipeline] Total  : 2203 frames
frame=0/2203  dets=6  total=6  0.0fps
frame=100/2203  dets=3  total=705  5.6fps
frame=200/2203  dets=5  total=1240  6.4fps
frame=300/2203

In [51]:
import subprocess, threading, time
REPO  = "/content/AAI-590-Capstone-group2"
VIDEO = f"{REPO}/output_ds_1.mp4"

with open(f"{REPO}/victor_deepstream/omniview_pipeline.py") as f:
    code = f.read()
code = code.replace("DISPLAY      = True",  "DISPLAY      = False")
code = code.replace("RECORD       = True",  "RECORD       = False")
code = code.replace("/home/logicpro09/omniview_ai/yolov8n_lisa_v1.1.pt",
                    f"{REPO}/victor_deepstream/yolov8n_lisa_v1.1.onnx")
code = code.replace("/home/logicpro09/omniview_ai/yolov8n.pt",
                    f"{REPO}/models/yolov8n.pt")
code = code.replace("/home/logicpro09/omniview_ai/output_ds_3_reenc.mp4", VIDEO)
code = code.replace("output_ds_3_reenc.mp4", VIDEO)
with open("/content/omniview_pipeline_patched.py", "w") as f:
    f.write(code)

proc = subprocess.Popen(
    ["python3", "-u", "/content/omniview_pipeline_patched.py"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
victor_started = threading.Event()
def _stream(p):
    for line in p.stdout:
        print(f"[Victor] {line}", end="", flush=True)
        if "fps" in line.lower() or "frame" in line.lower():
            victor_started.set()
threading.Thread(target=_stream, args=(proc,), daemon=True).start()
print("Waiting for Victor deepstream(up to 90s)...")
victor_started.wait(timeout=90)
print("Victor deep stream running! Run next cell now.")

Waiting for Victor deepstream(up to 90s)...
[Victor] WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
[Victor] JSON files -> /home/logicpro09/omniview_ai/runs/hud/ds_3
[Victor] Starting OmniView AI pipeline - dual model...
[Victor] LISA model: traffic sign detection
[Victor] COCO model: vehicle/pedestrian blind spot detection
[Victor] UDP stream -> 127.0.0.1:5055
[Victor] Video: /content/AAI-590-Capstone-group2/output_ds_1.mp4
[Victor] --------------------------------------------------
[Victor] Loading /content/AAI-590-Capstone-group2/victor_deepstream/yolov8n_lisa_v1.1.onnx for ONNX Runtime inference...
[Victor] Using ONNX Runtime 1.24.4 with CPUExecutionProvider
Victor deep stream running! Run next cell now.


In [52]:
import subprocess
REPO = "/content/AAI-590-Capstone-group2"

result = subprocess.run([
    "python3", f"{REPO}/omniview_e2e_live.py",
    "--no-bootstrap", "--mode", "udp",
    "--video",      f"{REPO}/output_ds_1.mp4",
    "--output",     "/content/omniview_live_hud_udp.mp4",
    "--gif-frames", "40", "--gif-fps", "10",
    "--udp-host",   "127.0.0.1",
    "--udp-port",   "5055",
    "--udp-wait",   "30",
    "--max-frames", "2203",
], capture_output=True, text=True)
proc.terminate()
print(result.stdout[-2000:])

  OmniView AI -- End-to-End Live HUD  v4.0
  AAI-590 Capstone Group 2 | USD
  Victor's omniview_pipeline.py loaded
     get_blind_spot_zones, draw_blind_spot_zones
     get_box_color, draw_alerts
     DetectionPayload, build_message
  Sunitha's VelocityTracker + INSTRUCTION_MAP loaded
  Tej's ds_pipeline.py referenced for Mode 3 (Jetson)
     Cannot import on Colab -- Jetson hardware required

  Mode   : UDP
  Video  : /content/AAI-590-Capstone-group2/output_ds_1.mp4
  JSON   : /content/AAI-590-Capstone-group2/runs/hud/ds_1
  Output : /content/omniview_live_hud_udp.mp4
[Pipeline] Video  : /content/AAI-590-Capstone-group2/output_ds_1.mp4  1920x1080 @ 30.0fps
[UDP] Listening on 127.0.0.1:5055
[Pipeline] UDP mode -- waiting 30.0s for first packet...
frame=0/2203  dets=2  total=2  0.0fps
frame=100/2203  dets=0  total=185  3.7fps
frame=200/2203  dets=3  total=382  4.1fps
frame=300/2203  dets=4  total=773  4.2fps
frame=400/2203  dets=4  total=1133  4.3fps
frame=500/2203  dets=4  total=1592  

In [53]:
import json, os
from IPython.display import display, HTML

perf_file = "/content/perf_file.json"
perf_udp  = "/content/perf_udp.json"

if os.path.exists(perf_udp) and os.path.exists(perf_file):
    p_path = perf_udp if os.path.getmtime(perf_udp) > os.path.getmtime(perf_file) else perf_file
elif os.path.exists(perf_udp):
    p_path = perf_udp
else:
    p_path = perf_file

with open(p_path) as f:
    p = json.load(f)

print(f"Showing: {p['mode']} mode")

rows = [
    ("section", "Pipeline & Render"),
    ("Mode",                        p["mode"]),
    ("Video resolution",            p["resolution"]),
    ("Total frames processed",      f"{p['total_frames']:,}"),
    ("Total runtime",               f"{p['elapsed_s']} s"),
    ("Average render FPS",          f"{p['avg_fps']} fps"),
    ("Avg frame render time",       f"{p['avg_render_ms']} ms"),
    ("Min frame render time",       f"{p['min_render_ms']} ms"),
    ("Max frame render time",       f"{p['max_render_ms']} ms"),
    ("Output MP4 size",             f"{p['mp4_kb']:,} KB"),
    ("section", "Detection Metrics"),
    ("Total detections drawn",      f"{p['total_dets']:,}"),
    ("Avg detections / frame",      f"{p['avg_dets_frame']}"),
    ("Frames with detections",      f"{p['frames_with_dets']:,}"),
    ("Detection rate",              f"{p['det_rate_pct']} %"),
    ("LISA detections (signs)",     f"{p['lisa_total']:,}"),
    ("COCO detections (vehicles)",  f"{p['coco_total']:,}"),
    ("section", "HUD & Alert Metrics"),
    ("Left blind spot alerts",      f"{p['left_alerts']:,}"),
    ("Right blind spot alerts",     f"{p['right_alerts']:,}"),
    ("Total blind spot alerts",     f"{p['left_alerts']+p['right_alerts']:,}"),
    ("Fast-approach events",        f"{p['fast_approach']:,}"),
    ("Unique objects tracked",      f"{p['tracked_objects']:,}"),
]

html = """<div style='background:#0A2A35;padding:24px;border-radius:12px;
            border:2px solid #41AEBD;font-family:monospace;max-width:700px'>
  <h2 style='color:#41AEBD;text-align:center;letter-spacing:3px;margin-bottom:4px'>
      OmniView AI — HUD Performance Report</h2>
  <p style='text-align:center;color:#7FB8C8;font-size:12px;margin-top:0'>
      Mode: {mode} &nbsp;|&nbsp; {res} &nbsp;|&nbsp;
      {frames} frames &nbsp;|&nbsp; {rt}s runtime</p>
  <table style='border-collapse:collapse;width:100%;font-size:13px'>
    <tr style='background:#41AEBD;color:#0A2A35'>
      <th style='padding:8px 14px;text-align:left'>Metric</th>
      <th style='padding:8px 14px;text-align:right'>Value</th></tr>""".format(
    mode=p["mode"], res=p["resolution"],
    frames=f"{p['total_frames']:,}", rt=p["elapsed_s"])

section_colors = {
    "Pipeline & Render":   "#41AEBD",
    "Detection Metrics":   "#4CAF82",
    "HUD & Alert Metrics": "#F5A623",
}
row_idx = 0
for item in rows:
    if item[0] == "section":
        col = section_colors.get(item[1], "#41AEBD")
        html += f"<tr style='background:{col}22'><td colspan='2' style='padding:8px 14px;color:{col};font-weight:bold;font-size:12px'>▸ {item[1]}</td></tr>"
        row_idx = 0
    else:
        k, v = item
        bg = "#0D3D50" if row_idx % 2 == 0 else "#0A2A35"
        html += f"<tr style='background:{bg}'><td style='padding:6px 14px;color:#D6EEF3'>{k}</td><td style='padding:6px 14px;text-align:right;color:#F4DE3A;font-weight:bold'>{v}</td></tr>"
        row_idx += 1

html += "</table></div>"
display(HTML(html))

Showing: UDP mode
